# Train

> GPUs pulling all the weight

In [ ]:
#| default_exp train

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, torch.nn as nn, torch.nn.functional as F, lightning.pytorch as pl, warnings, math

from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts
from torch.utils.data import DataLoader, TensorDataset

from wavefm.patchtst import PatchTFTSimple
from wavefm.loss import FocalLoss, KLDivLoss, CrossEntropyLoss
from tqdm import tqdm
from wavefm.survival import HypnogramEncoder

## Supervised Training with Linear Probing

### PatchTFT Masked Autoregression SS Prediction

In [ ]:
#| export
class PatchTFTSleepStage(pl.LightningModule):
    def __init__(self, 
                 learning_rate, # desired learning rate, initial learning rate in if one_cycle_scheduler
                 train_size, # the training data size (for one_cycle_scheduler=True)
                 batch_size, # the batch size (for one_cycle_scheduler=True)
                 n_gpus, # number of gpus to use
                 linear_probing_head, # model head to train
                 preloaded_model, # loaded pretrained model to use for linear probing
                 metrics={}, # metrics to calculate
                 fine_tune=False, # indicator to finetune encoder model or perform linear probing and freeze encoder weights
                 loss_fxn='CrossEntropy', # loss function to use, can be CrossEntropy or FocalLoss
                 class_weights=None, # weights of classes to use in CE loss fxn
                 gamma=2., # for focal loss
                 label_smoothing=0, # label smoothing for cross entropy loss
                 y_padding_mask=-100, # padded value that was added to target and indice to ignore when computing loss
                 epochs=100, # number of epochs for one_cycle_scheduler
                 weight_decay=0., # weight decay for Adam optimizer
                 use_weight_decay_scheduler=False, # use a weight decay scheduler
                 final_weight_decay=0.01, # final weight decay for the weight decay scheduler
                 optimizer_type='Adam', # optimizer to use, 'Adam' or 'AdamW'
                 scheduler_type='OneCycle', # scheduler to use, 'OneCycle' or 'CosineAnnealingWarmRestarts'
                 scheduler_kwargs={}, # kwargs for the scheduler
                 ):
        super().__init__()
        self.encoder = preloaded_model
        assert loss_fxn.lower() in ['crossentropy', 'focalloss', 'kldivloss'], "loss_fxn must be either CrossEntropy or FocalLoss or KLDivLoss"
        
        self.scheduler_type = scheduler_type
        if self.scheduler_type is not None:
            assert self.scheduler_type.lower() in ['onecycle', 'cosineannealingwarmrestarts'], "scheduler must be either OneCycle, CosineAnnealingWarmRestarts, or None"
        self.weight_decay = weight_decay
        self.label_smoothing = label_smoothing
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.train_size = train_size
        self.batch_size = batch_size * n_gpus
        self.ipe = self.train_size//self.batch_size
        self.metrics = nn.ModuleDict(metrics)
        self.y_padding_mask = y_padding_mask
        self.ipe = self.train_size//self.batch_size
        self.class_weights = class_weights
        self.loss_fxn = loss_fxn
        self.gamma = gamma
        self.fine_tune = fine_tune
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.final_weight_decay = final_weight_decay
        self.scheduler_kwargs = scheduler_kwargs

        if not self.fine_tune:
            self.encoder.freeze() # freeze the encoder weights (sets to eval mode)
            if hasattr(self.encoder, 'pretrain'):
                setattr(self.encoder, 'pretrain', False)
                setattr(self.encoder.model, 'pretrain', False)
        self.feedforward = linear_probing_head 
        self.optimizer_type = optimizer_type
        
        self.save_hyperparameters(ignore=['linear_probing_head', 'preloaded_model'])

    def forward(self, x):
        """
        In: [bs, n_channels, seq_len]
        """
        x = self.encoder(x) # [bs, n_channels, d_model, n_ffts/n_patches]
        if isinstance(x, tuple):
            x = x[0]
        x = self.feedforward(x) # [bs, n_classes, pred_len_seconds]
        return x
    
    def on_train_batch_start(self, batch, batch_idx):
        # update weight decay
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        x,y, *idx = batch
        x = self.encoder(x)
        if isinstance(x, tuple):
            x = x[0]
        preds = self.feedforward(x)
        return preds, y
    
    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        x, y, *idx = batch
        x = self.encoder(x)
        if isinstance(x, tuple):
            x = x[0]
        x = self.feedforward(x) 
        if x.is_nested:
            x = x.to_padded_tensor(padding=0)
        if y.is_nested:
            y = y.to_padded_tensor(padding=self.y_padding_mask, output_size=(x.shape[0], x.shape[2]))
        if self.loss_fxn.lower() == 'kldivloss':
            loss = KLDivLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, ignore_index=self.y_padding_mask)(x,y)
        else:
            ce_loss = CrossEntropyLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, label_smoothing=self.label_smoothing, ignore_index=self.y_padding_mask)
            focal_loss = FocalLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, gamma=self.gamma, ignore_index=self.y_padding_mask)
            loss = ce_loss(x,y) if self.loss_fxn == 'CrossEntropy' else focal_loss(x,y)
            self.log('train_ce_loss' if self.loss_fxn != 'CrossEntropy' else 'train_focal_loss', ce_loss(x,y) if self.loss_fxn != 'CrossEntropy' else focal_loss(x,y), prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        loss = loss.to(x.device)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        
        # debugging
        if batch_idx == 0:
            print("Head parameters and requires_grad:")
            for name, param in self.feedforward.named_parameters():
                print(name, param.requires_grad, param.grad.abs().mean() if param.grad is not None else None)
            print("Head output shape:", x.shape)
            print("Unique targets:", torch.unique(y, return_counts=True))
            print("Unique predicted classes:", torch.unique(torch.argmax(x, dim=1), return_counts=True))
            print("Predicted classes (first sample):", torch.argmax(x[0], dim=0)[:20])
            print("Target classes (first sample):", y[0][:20])
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y, *idx = batch
        x = self.encoder(x) # val should be in eval mode
        if isinstance(x, tuple):
            x = x[0]
        x = self.feedforward(x)
        if x.is_nested:
            x = x.to_padded_tensor(padding=0)
        if y.is_nested:
            y = y.to_padded_tensor(padding=self.y_padding_mask, output_size=(x.shape[0], x.shape[2]))
        if self.loss_fxn.lower() == 'kldivloss':
            loss = KLDivLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, ignore_index=self.y_padding_mask)(x,y)
        else:
            ce_loss = CrossEntropyLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, label_smoothing=self.label_smoothing, ignore_index=self.y_padding_mask)
            focal_loss = FocalLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, gamma=self.gamma, ignore_index=self.y_padding_mask)
            loss = ce_loss(x,y) if self.loss_fxn == 'CrossEntropy' else focal_loss(x,y)
            self.log('val_ce_loss' if self.loss_fxn != 'CrossEntropy' else 'val_focal_loss', ce_loss(x,y) if self.loss_fxn != 'CrossEntropy' else focal_loss(x,y), prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        #loss = loss.to(x.device)
        self.log('val_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        x_probs = torch.softmax(x, dim=1)
        for metric in self.metrics:
            self.metrics[metric].update(x_probs, y)

    def on_validation_epoch_start(self):
        torch.cuda.empty_cache()
    
    def on_validation_epoch_end(self):
        for name, metric in self.metrics.items():
            self.log(f'val_{name}', metric.compute(), prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
            metric.reset()
        
    def configure_optimizers(self):
        param_groups = [ # exclude bias and layer norm parameters from weight decay
           {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            },
        ]
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, total_steps=self.trainer.estimated_stepping_batches, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

### PatchTFT Single Outcome Prediction

In [ ]:
#| export
class PatchTFTSingleOutcomeLightning(pl.LightningModule): #encoder for linear probing head that predict EDS
    def __init__(self, #constructor that contains or calls the main dataset setup code
                linear_probing_head, # model head to linear probe/train
                learning_rate, # desired learning rate, initial learning rate in if one_cycle_scheduler
                train_size, # the training data size (for one_cycle_scheduler=True)
                batch_size, # the batch size (for one_cycle_scheduler=True)
                n_gpus, # number of GPUs
                preloaded_model, # loaded pretrained model to use for linear probing
                metrics={}, # name:function for metrics to log
                fine_tune=False, # indicator to fine tune encoder or freeze encoder weights and perform linear probing
                class_weights=None, # weights of classes to use in CE loss fxn
                epochs=100, # number of epochs for one_cycle_scheduler
                scheduler_type='OneCycle',
                optimizer_type='AdamW',
                weight_decay=0., # weight decay for Adam optimizer
                use_weight_decay_scheduler=False, # use a weight decay scheduler
                final_weight_decay=0.01, # final weight decay for the weight decay scheduler
                scheduler_kwargs={}, # kwargs for the scheduler
                transforms=None, # transforms to apply to the data
                mixup_callback=None, # mixup callback to apply to the data
                regression=False, # whether the outcome is regression or classification
                loss_func=None, # loss function to use
                ):
        super().__init__()
        self.encoder = preloaded_model
        self.scheduler_type = scheduler_type
        if self.scheduler_type is not None:
            assert self.scheduler_type.lower() in ['onecycle', 'cosineannealingwarmrestarts'], "scheduler must be either OneCycle, CosineAnnealingWarmRestarts, or None"
        self.weight_decay = weight_decay
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.train_size = train_size
        self.batch_size = batch_size * n_gpus
        self.ipe = self.train_size//self.batch_size
        self.metrics = nn.ModuleDict(metrics)
        self.ipe = self.train_size//self.batch_size
        self.class_weights = class_weights
        self.fine_tune = fine_tune
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.scheduler_kwargs = scheduler_kwargs
        self.final_weight_decay = final_weight_decay
        self.optimizer_type = optimizer_type
        self.transforms = transforms
        self.mixup_callback = mixup_callback
        self.regression = regression
        self.loss_func = loss_func
        if not self.fine_tune:
            self.encoder.freeze() # freeze the encoder weights (sets to eval mode)
            if hasattr(self.encoder, 'pretrain'):
                setattr(self.encoder, 'pretrain', False)
                setattr(self.encoder.model, 'pretrain', False)
        # Adjust the output layer for binary classification
        self.feedforward = linear_probing_head # model head to linear probe/train
        self.save_hyperparameters(ignore=['linear_probing_head', 'preloaded_model'])

    def forward(self, x):
        x = self.encoder(x) # [bs, n_channels, d_model, n_ffts/n_patches]
        if isinstance(x, tuple):
            # contrastive model
            x = x[0]
        if torch.isnan(x).any():
            warnings.warn("NaN values in input to feedforward layer")
        x = self.feedforward(x) # [bs, n_classes, pred_len_seconds]
        return x # Ensure the output is of shape [batch_size]
    
    def on_train_batch_start(self, batch, batch_idx):
        # update weight decay
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd

    def predict_step(self, batch, batch_idx, dataloader_idx=0): #does the order matter? should it be after test_step()
        x,y = batch
        preds = self(x)
        return preds, y

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        if self.transforms is not None:
            batch = self.transforms(batch)
        if self.mixup_callback is not None:
            batch = self.mixup_callback(batch)
        x, y = batch
        x = self(x)  # Ensure output is [batch_size]
        if self.regression:
            loss = nn.MSELoss() if self.loss_func is None else self.loss_func
            loss_val = loss(x.squeeze(), y.squeeze().float())
        else:
            loss = nn.BCEWithLogitsLoss(pos_weight=self.class_weights.to(x.device) if self.class_weights is not None else None) if self.loss_func is None else self.loss_func
            loss_val = loss(x.squeeze(),y.squeeze().float())
        self.log('train_loss', loss_val, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        return loss_val

    def validation_step(self, batch, batch_idx):
        x, y = batch
        x = self(x)  # Ensure output is [batch_size]
        if self.regression:
            loss = nn.MSELoss() if self.loss_func is None else self.loss_func
            loss_val = loss(x.squeeze(), y.squeeze().float())
        else:
            loss = nn.BCEWithLogitsLoss(pos_weight=self.class_weights.to(x.device) if self.class_weights is not None else None) if self.loss_func is None else self.loss_func
            loss_val = loss(x.squeeze(),y.squeeze().float())
        self.log('val_loss', loss_val, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        
        x_probs = torch.sigmoid(x)
        for metric in self.metrics:
            self.metrics[metric].update(x_probs.squeeze(), y.squeeze().long())
    
    def on_validation_epoch_end(self):
        all_score = 0
        for name, metric in self.metrics.items():
            metric_val = metric.compute()
            all_score += metric_val
            self.log(f'val_{name}', metric_val, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
            metric.reset()
        self.log('val_all_score', all_score, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
    
    def on_validation_epoch_start(self): #start of a validation epoch during the training or evaluation of a model in a deep learning framework
        #do you need to end it?
        torch.cuda.empty_cache() #releases all unused memory cached by the CUDA driver 
        #for the PyTorch process to potentially reduce the memory footprint on the GPU

    def configure_optimizers(self):
        param_groups = [ # exclude bias and layer norm parameters from weight decay
           {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            },
        ]
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, total_steps=self.trainer.estimated_stepping_batches, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

In [ ]:
#| export
class PatchTFTSingleOutcomeHypnogramLightning(pl.LightningModule): #encoder for linear probing head that predict EDS
    def __init__(self, #constructor that contains or calls the main dataset setup code
                linear_probing_head, # model head to linear probe/train
                learning_rate, # desired learning rate, initial learning rate in if one_cycle_scheduler
                train_size, # the training data size (for one_cycle_scheduler=True)
                batch_size, # the batch size (for one_cycle_scheduler=True)
                n_gpus, # number of GPUs
                preloaded_model, # loaded pretrained model to use for linear probing
                patch_len,
                d_model,
                metrics={}, # name:function for metrics to log
                hypnogram_index=7,
                signal_only=False,
                zero_hypnogram=False,
                hypnogram_only=False,
                fine_tune=False, # indicator to fine tune encoder or freeze encoder weights and perform linear probing
                class_weights=None, # weights of classes to use in CE loss fxn
                epochs=100, # number of epochs for one_cycle_scheduler
                scheduler_type='OneCycle',
                optimizer_type='AdamW',
                weight_decay=0., # weight decay for Adam optimizer
                use_weight_decay_scheduler=False, # use a weight decay scheduler
                final_weight_decay=0.01, # final weight decay for the weight decay scheduler
                scheduler_kwargs={}, # kwargs for the scheduler
                transforms=None, # transforms to apply to the data
                mixup_callback=None, # mixup callback to apply to the data
                ):
        super().__init__()
        self.encoder = preloaded_model
        self.scheduler_type = scheduler_type
        if self.scheduler_type is not None:
            assert self.scheduler_type.lower() in ['onecycle', 'cosineannealingwarmrestarts'], "scheduler must be either OneCycle, CosineAnnealingWarmRestarts, or None"
        self.weight_decay = weight_decay
        self.learning_rate = learning_rate
        self.hypnogram_index = torch.tensor([hypnogram_index])
        self.x_index = torch.tensor(list(range(self.hypnogram_index)))
        self.epochs = epochs
        self.train_size = train_size
        self.batch_size = batch_size * n_gpus
        self.ipe = self.train_size//self.batch_size
        self.metrics = nn.ModuleDict(metrics)
        self.ipe = self.train_size//self.batch_size
        self.class_weights = class_weights
        self.fine_tune = fine_tune
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.scheduler_kwargs = scheduler_kwargs
        self.final_weight_decay = final_weight_decay
        self.optimizer_type = optimizer_type
        self.transforms = transforms
        self.mixup_callback = mixup_callback
        self.signal_only = signal_only
        self.zero_hypnogram = zero_hypnogram
        self.hypnogram_only = hypnogram_only
        assert not (self.signal_only and self.hypnogram_only), "Signal only and hypnogram only cannot be True at the same time"

        if not self.fine_tune:
            self.encoder.freeze() # freeze the encoder weights (sets to eval mode)
            if hasattr(self.encoder, 'pretrain'):
                setattr(self.encoder, 'pretrain', False)
                setattr(self.encoder.model, 'pretrain', False)
        # Adjust the output layer for binary classification
        if not self.signal_only:
            self.hypnogram_encoder = HypnogramEncoder(
                patch_len=patch_len,
                num_categories=6,
                d_model=d_model
            )
        else:
            self.hypnogram_encoder = nn.Identity()
        self.feedforward = linear_probing_head # model head to linear probe/train
        self.save_hyperparameters(ignore=['linear_probing_head', 'preloaded_model'])

    def forward(self, x):
        if x.is_nested:
            hyps = []
            x_vals = []
            for x_i in x:
                if self.zero_hypnogram:
                    hyps.append(torch.zeros_like(x_i.index_select(dim=0, index=self.hypnogram_index.to(x.device)).transpose(0, 1)))
                else:
                    hyps.append(x_i.index_select(dim=0, index=self.hypnogram_index.to(x.device)).transpose(0, 1))
                x_vals.append(x_i.index_select(dim=0, index=self.x_index.to(x.device)).transpose(0, 1))
            x = torch.nested.as_nested_tensor(x_vals, layout=torch.jagged)
            hypnogram = torch.nested.as_nested_tensor(hyps, layout=torch.jagged)
            x = x.transpose(1, 2)
            hypnogram = hypnogram.transpose(1, 2)
        else:
            if self.zero_hypnogram:
                hypnogram = torch.zeros_like(torch.index_select(x, dim=1, index=self.hypnogram_index.to(x.device)))
            else:
                hypnogram = torch.index_select(x, dim=1, index=self.hypnogram_index.to(x.device))
            x = torch.index_select(x, dim=1, index=self.x_index.to(x.device))
        hypnogram = hypnogram.to(torch.int64)
        x = self.encoder(x) # [bs, n_channels, d_model, n_ffts/n_patches]
        hypnogram_enc = self.hypnogram_encoder(hypnogram)
        if isinstance(x, tuple):
            # contrastive model
            x = x[0]
        if torch.isnan(x).any():
            warnings.warn("NaN values in input to feedforward layer")
        
        if not self.hypnogram_only and not self.signal_only:
            if x.is_nested:
                x_list = []
                for x_i, h_i in zip(x, hypnogram_enc): # ex torch.Size([6, 1, 256, j14]), the jagged dimension should be 1
                    x_list.append(torch.cat([x_i, h_i], dim=0).permute(2,0,1)) # patch dim (jagged), channels, d_model
                x_cat = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
                x_cat = x_cat.permute(0, 2, 3, 1) # bs, channels, d_model, patch dim (jagged)
            else:
                x_cat = torch.cat([x, hypnogram_enc], dim=1) # stack in channel dimension
        elif self.signal_only:
            x_cat = x
        else:
            x_cat = hypnogram_enc
        x_cat = self.feedforward(x_cat) # [bs, n_classes, pred_len_seconds]
        return x_cat # Ensure the output is of shape [batch_size]
    
    def on_train_batch_start(self, batch, batch_idx):
        # update weight decay
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd

    def predict_step(self, batch, batch_idx, dataloader_idx=0): #does the order matter? should it be after test_step()
        x,y = batch
        preds = self(x)
        return preds, y

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        if self.transforms is not None:
            batch = self.transforms(batch)
        if self.mixup_callback is not None:
            batch = self.mixup_callback(batch)
        x, y = batch
        x = self(x)
        loss = nn.BCEWithLogitsLoss(pos_weight=self.class_weights.to(x.device) if self.class_weights is not None else None)
        loss_val = loss(x, y.float())
        self.log('train_loss', loss_val, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        return loss_val

    def validation_step(self, batch, batch_idx):
        x, y = batch
        x = self(x)
        loss = nn.BCEWithLogitsLoss(pos_weight=self.class_weights.to(x.device) if self.class_weights is not None else None)
        loss_val = loss(x, y.float())
        self.log('val_loss', loss_val, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        
        x_probs = torch.sigmoid(x)
        for metric in self.metrics:
            self.metrics[metric].update(x_probs, y.long())
    
    def on_validation_epoch_end(self):
        for name, metric in self.metrics.items():
            metric_val = metric.compute()
            self.log(f'val_{name}', metric_val, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
            metric.reset()
    
    def on_validation_epoch_start(self): #start of a validation epoch during the training or evaluation of a model in a deep learning framework
        #do you need to end it?
        torch.cuda.empty_cache() #releases all unused memory cached by the CUDA driver 
        #for the PyTorch process to potentially reduce the memory footprint on the GPU

    def configure_optimizers(self):
        param_groups = [ # exclude bias and layer norm parameters from weight decay
            {
                'params': (p for n, p in self.hypnogram_encoder.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.hypnogram_encoder.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            }, {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            },
        ]
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, total_steps=self.trainer.estimated_stepping_batches, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

In [ ]:
hypnogram_index = 7
import torch
hypnogram_index = torch.tensor([hypnogram_index])
x_index = torch.tensor(list(range(hypnogram_index)))

## Full Supervised Training

### PatchTFT Sleep Stage Supervised Training

In [ ]:
#| export
class PatchTFTSupervised(pl.LightningModule): #encoder for linear probing head that predict EDS
    def __init__(self, #constructor that contains or calls the main dataset setup code
                encoder_kwargs, # args to initialize the PatchTFT encoder
                linear_probing_head, # model head to linear probe/train
                learning_rate, # desired learning rate, initial learning rate in if one_cycle_scheduler
                train_size, # the training data size (for one_cycle_scheduler=True)
                batch_size, # the batch size (for one_cycle_scheduler=True)
                n_gpus, # number of GPUs
                num_nodes=1, # number of nodes
                metrics={}, # name:function for metrics to log
                class_weights=None, # weights of classes to use in CE loss fxn
                epochs=100, # number of epochs for one_cycle_scheduler
                scheduler_type='OneCycle',
                optimizer_type='AdamW',
                weight_decay=0., # weight decay for Adam optimizer
                use_weight_decay_scheduler=False, # use a weight decay scheduler
                final_weight_decay=0.01, # final weight decay for the weight decay scheduler
                scheduler_kwargs={}, # kwargs for the scheduler
                transforms=None, # transforms to apply to the data
                mixup_callback=None, # mixup callback to apply to the data
                ):
        super().__init__()
        self.scheduler_type = scheduler_type
        if self.scheduler_type is not None:
            assert self.scheduler_type.lower() in ['onecycle', 'cosineannealingwarmrestarts'], "scheduler must be either OneCycle, CosineAnnealingWarmRestarts, or None"
        self.weight_decay = weight_decay
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.train_size = train_size
        self.batch_size = batch_size*n_gpus*num_nodes
        self.ipe = self.train_size//self.batch_size
        self.metrics = nn.ModuleDict(metrics)
        self.ipe = self.train_size//self.batch_size
        self.class_weights = class_weights
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.scheduler_kwargs = scheduler_kwargs
        self.final_weight_decay = final_weight_decay
        self.optimizer_type = optimizer_type
        self.transforms = transforms
        self.mixup_callback = mixup_callback
        # Adjust the output layer for binary classification
        self.encoder = PatchTFTSimple(**encoder_kwargs)
        self.feedforward = linear_probing_head # model head to linear probe/train
        self.save_hyperparameters(ignore=['linear_probing_head', 'preloaded_model'])

    def forward(self, x):
        if torch.isnan(x).any():
            warnings.warn("NaN values in input to encoder layer")
        x = self.encoder(x) # [bs * n_channels, n_patches, d_model]
        if isinstance(x, tuple):
            # contrastive model
            x = x[0]
        if torch.isnan(x).any():
            warnings.warn("NaN values in input to feedforward layer")
        x = self.feedforward(x) # [bs, n_classes, pred_len_seconds]
        if torch.isnan(x).any():
            warnings.warn("NaN values in output of feedforward layer")
        return x # Ensure the output is of shape [batch_size]
    
    def on_train_batch_start(self, batch, batch_idx):
        # update weight decay
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd

    def predict_step(self, batch, batch_idx, dataloader_idx=0): #does the order matter? should it be after test_step()
        x,y, *idx = batch
        preds = self(x)
        return preds, y

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        if self.transforms is not None:
            batch = self.transforms(batch)
        if self.mixup_callback is not None:
            batch = self.mixup_callback(batch)
        x, y, *idx = batch
        x = self(x)  # Ensure output is [batch_size]
        if torch.isnan(y).any():
            warnings.warn(f"NaN values in training targets at batch index {batch_idx} (skipping batch)")
            return None
        elif torch.isnan(x).any():
            warnings.warn(f"NaN values in training predictions at batch index {batch_idx} (skipping batch)")
            return None
        else:
            loss = nn.BCEWithLogitsLoss(pos_weight=self.class_weights.to(x.device) if self.class_weights is not None else None)
            loss_val = loss(x,y.float())
            self.log('train_loss', loss_val, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
            return loss_val

    def validation_step(self, batch, batch_idx):
        x, y, *idx = batch
        x = self(x)  # Ensure output is [batch_size]

        if torch.isnan(y).any() or torch.isnan(x).any():
            if torch.isnan(y).any():
                warnings.warn(f"NaN values in validation targets at batch index {batch_idx} (skipping metric update)")
            if torch.isnan(x).any():
                warnings.warn(f"NaN values in validation predictions at batch index {batch_idx} (skipping metric update)")
        else:
            loss = nn.BCEWithLogitsLoss(pos_weight=self.class_weights.to(x.device) if self.class_weights is not None else None)
            loss_val = loss(x,y.float())
            self.log('val_loss', loss_val, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
            x_probs = torch.sigmoid(x)
        if not torch.isnan(x_probs).any():
            for metric in self.metrics:
                self.metrics[metric].update(x_probs, y.long())
        else:
            warnings.warn(f"NaN values after sigmoid on validation predictions at batch index {batch_idx} (skipping metric update)")
    
    def on_validation_epoch_end(self):
        for name, metric in self.metrics.items():
            metric_val = metric.compute()
            self.log(f'val_{name}', metric_val, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
            metric.reset()
    
    def on_validation_epoch_start(self): #start of a validation epoch during the training or evaluation of a model in a deep learning framework
        #do you need to end it?
        torch.cuda.empty_cache() #releases all unused memory cached by the CUDA driver 
        #for the PyTorch process to potentially reduce the memory footprint on the GPU

    def configure_optimizers(self):
        param_encoder = [ # exclude bias and layer norm parameters from weight decay
            {
                'params': (p for n, p in self.encoder.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.encoder.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            }
        ]

        param_feedforward = [ # exclude bias and layer norm parameters from weight decay
            {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            },
        ]
        param_groups = param_encoder + param_feedforward
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, total_steps=self.trainer.estimated_stepping_batches, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

## Miscellaneous

In [ ]:
#| export
class LinearProbingCallback(pl.Callback):
    def __init__(self, lp_model, lp_epochs, lp_train_dataloader, lp_val_dataloader, lp_class_weights, learning_rate, weight_decay=0., eval_frequency=5):
        super().__init__()
        self.eval_frequency = eval_frequency
        self.lp_model = lp_model
        self.lp_epochs = lp_epochs
        self.lp_train_dataloader = lp_train_dataloader
        self.lp_val_dataloader = lp_val_dataloader
        self.lp_class_weights = lp_class_weights
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        
    def on_validation_epoch_end(self, trainer, pl_module):
        if self.lp_model is not None:
            if (trainer.current_epoch+1) % self.eval_frequency == 0:
                # Get embeddings using current encoder
                pl_module.model.context_encoder.eval()
                self.lp_model = self.lp_model.to(pl_module.device)

                optimizer = torch.optim.AdamW(self.lp_model.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)

                scheduler = torch.optim.lr_scheduler.OneCycleLR(
                    optimizer,
                    max_lr=0.001,
                    epochs=self.lp_epochs,
                    steps_per_epoch=len(self.lp_train_dataloader),
                    pct_start=0.2
                )

                criterion = nn.CrossEntropyLoss(weight=self.lp_class_weights.to(pl_module.device) if self.lp_class_weights is not None else None)
                train_losses = []
                val_losses = []

                epoch_pbar = tqdm(range(self.lp_epochs), desc='Linear Probe Training')
                for epoch in epoch_pbar:
                    train_epoch_losses = []
                    self.lp_model.train()
                    train_pbar = tqdm(self.lp_train_dataloader, 
                                desc=f'Training Epoch {epoch+1}/{self.lp_epochs}',
                                leave=False)
                    for batch in train_pbar:
                        x, y = batch
                        x, y = x.to(pl_module.device), y.to(pl_module.device)
                        with torch.set_grad_enabled(True):
                            emb = pl_module.model.context_encoder(x).detach()
                            emb = F.layer_norm(emb, emb.shape[-1:])
                            
                            optimizer.zero_grad()
                            outputs = self.lp_model(emb)
                            loss = criterion(outputs, y)
                            train_epoch_losses.append(loss.item())
                            loss.backward()
                            torch.nn.utils.clip_grad_norm_(self.lp_model.parameters(), 1.0)
                            optimizer.step()
                            scheduler.step()
                        train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
                    
                    val_epoch_losses = []
                    self.lp_model.eval()
                    val_pbar = tqdm(self.lp_val_dataloader, 
                            desc=f'Validation Epoch {epoch+1}/{self.lp_epochs}',
                            leave=False)
                    for batch in val_pbar:
                        x, y = batch
                        x, y = x.to(pl_module.device), y.to(pl_module.device)
                        with torch.no_grad():
                            emb = pl_module.model.context_encoder(x)
                            emb = F.layer_norm(emb, emb.shape[-1:])
                            outputs = self.lp_model(emb)
                            loss = criterion(outputs, y)
                            val_epoch_losses.append(loss.item())
                        val_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
                    avg_train_loss = sum(train_epoch_losses) / len(train_epoch_losses)
                    avg_val_loss = sum(val_epoch_losses) / len(val_epoch_losses)
                    train_losses.append(avg_train_loss)
                    val_losses.append(avg_val_loss)
                    epoch_pbar.set_postfix({
                        'train_loss': f'{avg_train_loss:.4f}',
                        'val_loss': f'{avg_val_loss:.4f}'
                    })
                final_train_loss = sum(train_losses) / len(train_losses)
                final_val_loss = sum(val_losses) / len(val_losses)
                pl_module.log("linear_probe_train_loss", final_train_loss, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
                pl_module.log("linear_probe_val_loss", final_val_loss, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
                pl_module.model.context_encoder.train()
            else:
                pl_module.log("linear_probe_train_loss", float('inf'), prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
                pl_module.log("linear_probe_val_loss", float('inf'), prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)

In [ ]:
#| notest
class EncoderTestFreezingWeights(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Linear(750,512)
    def forward(self, x):
        x = self.encoder(x)
        return x

class DecoderTest(pl.LightningModule):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.encoder.freeze()
        self.decoder = nn.Linear(512,750)
        
    def training_step(self, batch, batch_idx):
        print("Training step:")
        print(f"Encoder training mode: {self.encoder.training}")
        print(f"Decoder training mode: {self.decoder.training}")
        print(f"Model training mode: {self.training}")
        return torch.tensor(0.0, requires_grad=True)
        
    def validation_step(self, batch, batch_idx):
        print("Validation step:")
        print(f"Encoder training mode: {self.encoder.training}")
        print(f"Decoder training mode: {self.decoder.training}")
        print(f"Model training mode: {self.training}")
        
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.001)
        
    def train_dataloader(self):
        # Create a simple dummy dataset
        return DataLoader(TensorDataset(torch.randn(10, 750), torch.randn(10, 750)), batch_size=2)
        
    def val_dataloader(self):
        # Create a simple dummy dataset
        return DataLoader(TensorDataset(torch.randn(10, 750), torch.randn(10, 750)), batch_size=2)

# Create a trainer and properly run validation
encoder = EncoderTestFreezingWeights()
decoder = DecoderTest(encoder)
trainer = pl.Trainer(fast_dev_run=True)
trainer.fit(decoder)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()